# CareLoop Clinics — No-Show Guard

### Appointment No-Show Analysis and Risk Prediction

**Project Overview**

CareLoop Clinics — No-Show Guard is a data analytics and machine learning project designed to understand appointment no-shows and help clinics identify patients who may need additional follow-up.

The project combines Python, SQL, Machine Learning and Generative AI to analyse appointment records, identify patterns associated with missed appointments, predict no-show risk and classify patient follow-up notes.

**Project Objectives**

- Clean and prepare appointment and clinic datasets.
- Analyse appointment no-show patterns using SQL.
- Build and evaluate machine learning classification models.
- Use Gemini to classify patient follow-up notes.
- Develop practical recommendations to improve appointment attendance.

**Tools and Technologies**

Python · Pandas · NumPy · SQL · SQLite · Scikit-learn · Google Gemini API · Google Colab

**Project Structure**

1. Part 1 — Python and Data Foundations
2. Part 2 — SQL Analysis
3. Part 3 — Machine Learning
4. Part 4 — Generative AI Analysis
5. Final Business Recommendations

# Part 1 — Python and Data Foundations

## 1.1 Data Loading and Initial Inspection

In this section, the raw appointment dataset is loaded and examined using Python. The objective is to understand the dataset structure, identify data-quality issues and prepare the records for further analysis.


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'rejected_rows.json', 'followup_notes.csv', 'clinics.csv', 'appointments.csv', 'sample_data']


In [ ]:

import csv
import json

# Location of the appointments file
file_path = "/content/appointments.csv"

# Read the CSV file
with open(file_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

    print("Column headers:")
    print(reader.fieldnames)

    print("\nTotal rows:", len(rows))


Column headers:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Total rows: 1020


In [ ]:

def parse_row(row):
    converted_row = row.copy()

    converted_row["age"] = int(row["age"])
    converted_row["fee_inr"] = int(row["fee_inr"])
    converted_row["prior_no_shows"] = int(row["prior_no_shows"])
    converted_row["distance_km"] = float(row["distance_km"])

    return converted_row


accepted = []
rejected = []

for row in rows:
    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError as error:
        rejected.append({
            "row": row,
            "error": str(error)
        })

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("Total processed:", len(accepted) + len(rejected))


Accepted rows: 988
Rejected rows: 32
Total processed: 1020


In [ ]:

with open("/content/rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")


Rejected rows saved successfully!


In [ ]:
import os
import json

# Check whether the file exists
print("File exists:", os.path.exists("/content/rejected_rows.json"))

# Read the saved JSON file
with open("/content/rejected_rows.json", "r", encoding="utf-8") as file:
    saved_rejected = json.load(file)

print("Rejected records in JSON:", len(saved_rejected))
print("First rejected record:")
print(saved_rejected[0])

File exists: True
Rejected records in JSON: 32
First rejected record:
{'row': {'appointment_id': 'A0050', 'patient_id': 'P117', 'clinic_id': 'C04', 'booked_on': '2025-01-16', 'appt_date': '2025-01-24', 'age': '41', 'gender': 'M', 'distance_km': '', 'fee_inr': '600', 'booking_channel': 'Walk-in', 'visit_type': 'Consultation', 'reminder_sent': 'No', 'prior_no_shows': '0', 'status': 'Attended'}, 'error': "could not convert string to float: ''"}


In [ ]:
print("Total original rows:", len(rows))
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

print("\nFirst rejected row:")
print(rejected[0])

Total original rows: 1020
Accepted rows: 988
Rejected rows: 32

First rejected row:
{'row': {'appointment_id': 'A0050', 'patient_id': 'P117', 'clinic_id': 'C04', 'booked_on': '2025-01-16', 'appt_date': '2025-01-24', 'age': '41', 'gender': 'M', 'distance_km': '', 'fee_inr': '600', 'booking_channel': 'Walk-in', 'visit_type': 'Consultation', 'reminder_sent': 'No', 'prior_no_shows': '0', 'status': 'Attended'}, 'error': "could not convert string to float: ''"}


### 1.2  Inspect and clean

In [ ]:
blank_age = 0
blank_distance = 0

for item in rejected:
    row = item["row"]

    if row["age"] == "":
        blank_age += 1

    if row["distance_km"] == "":
        blank_distance += 1

print("Blank age:", blank_age)
print("Blank distance:", blank_distance)

Blank age: 12
Blank distance: 20


In [ ]:

import pandas as pd

# Load the original appointments CSV file
df = pd.read_csv("/content/appointments.csv")

# Display the first 5 rows
print("First 5 rows:")
display(df.head())

'''
import pandas as pd imports the pandas library.



pd.read_csv() reads your original CSV file.



df is the variable that stores the dataset.



df.head() displays the first five records.



display() presents the data in a readable table.
'''


First 5 rows:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


'\nimport pandas as pd imports the pandas library.\n\n\n\npd.read_csv() reads your original CSV file.\n\n\n\ndf is the variable that stores the dataset.\n\n\n\ndf.head() displays the first five records.\n\n\n\ndisplay() presents the data in a readable table.\n'

In [ ]:

# Check the number of rows and columns
print("Dataset shape:")
print(df.shape)

# Check column names and data types
print("\nDataset information:")
df.info()

# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

# Count exact duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())


Dataset shape:
(1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patie

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())

Shape after removing duplicates: (1000, 14)
Remaining duplicate rows: 0


In [ ]:

# Check unique booking channel values before cleaning
print("Before cleaning:")
print(df["booking_channel"].unique())

# Remove extra spaces and standardize capitalization
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

# Check unique values after cleaning
print("\nAfter cleaning:")
print(df["booking_channel"].unique())

print("\nNumber of unique booking channels:",
      df["booking_channel"].nunique())


Before cleaning:
['Phone' 'Walk-in' 'App' ' walk-in' 'APP ' 'Walk-In' 'Phone ' 'PHONE'
 'app' ' App' 'walk-in' 'phone' 'Walk-in ' ' phone' 'WALK-IN' 'App ']

After cleaning:
['Phone' 'Walk-In' 'App']

Number of unique booking channels: 3


In [ ]:

# Convert appointment date
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

# Convert booking date
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"                         #errors="coerce" converts unreadable dates into NaT (pandas' missing datetime value) rather than stopping your program.
)

# Check the date data types
print(df[["appt_date", "booked_on"]].dtypes)

# Count missing dates
print("\nMissing appointment dates:",
      df["appt_date"].isna().sum())

print("Missing booking dates:",
      df["booked_on"].isna().sum())


appt_date    datetime64[ns]
booked_on    datetime64[ns]
dtype: object

Missing appointment dates: 6
Missing booking dates: 0


In [ ]:

# Remove rows where appointment date or status is missing
df = df.dropna(subset=["appt_date", "status"])

print("Final dataset shape:", df.shape)

print("\nRemaining missing appointment dates:",
      df["appt_date"].isna().sum())

print("Remaining missing statuses:",
      df["status"].isna().sum())


Final dataset shape: (990, 14)

Remaining missing appointment dates: 0
Remaining missing statuses: 0


### Data Cleaning Results

The appointment dataset was checked for invalid age values and missing numerical data.

- Five invalid age values were identified.
- The median age of 48.0 was used to replace invalid or missing age values.
- No missing age values remained after cleaning.
- No missing distance values remained after cleaning.

These checks help ensure that the dataset is suitable for subsequent SQL analysis and machine learning.

## 1.3 Data Validity, Missing Values and Outliers

In [ ]:

# Find ages below 1 or above 100
invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Number of invalid ages:", invalid_age.sum())

# Display invalid age values
print("\nInvalid age records:")
display(df.loc[invalid_age, ["appointment_id", "age"]])


Number of invalid ages: 5

Invalid age records:


,appointment_id,age
79,A0077,0.0
533,A0522,150.0
588,A0576,-4.0
798,A0781,250.0
896,A0878,199.0


In [ ]:

import numpy as np

# Replace invalid ages with NaN
df.loc[invalid_age, "age"] = np.nan

print("Invalid ages replaced with NaN.")
print("Total missing ages now:", df["age"].isna().sum())


Invalid ages replaced with NaN.
Total missing ages now: 17


In [ ]:
#The median is the middle value when numbers are arranged in order. It is useful because extreme values have less influence on it than on the mean.
# Calculate median age
age_median = df["age"].median()

print("Median age:", age_median)

# Fill missing ages with median
df["age"] = df["age"].fillna(age_median)

# Convert age to integer
df["age"] = df["age"].round().astype(int)

print("\nMissing ages remaining:", df["age"].isna().sum())


Median age: 48.0

Missing ages remaining: 0


In [ ]:

# Calculate median distance
distance_median = df["distance_km"].median()

print("Median distance:", distance_median)

# Fill missing distances
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("\nMissing distances remaining:",
      df["distance_km"].isna().sum())


Median distance: 8.25

Missing distances remaining: 0


In [ ]:
print("Invalid ages identified:", invalid_age.sum())

print("Median age used:", age_median)

print("Missing ages remaining:", df["age"].isna().sum())

print("Missing distances remaining:", df["distance_km"].isna().sum())

Invalid ages identified: 5
Median age used: 48.0
Missing ages remaining: 0
Missing distances remaining: 0


In [ ]:

# Calculate quartiles for distance
distance_q1 = df["distance_km"].quantile(0.25)
distance_q3 = df["distance_km"].quantile(0.75)

# Calculate IQR
distance_iqr = distance_q3 - distance_q1

# Calculate lower and upper limits
distance_lower = distance_q1 - (1.5 * distance_iqr)
distance_upper = distance_q3 + (1.5 * distance_iqr)

print("Distance Q1:", distance_q1)
print("Distance Q3:", distance_q3)
print("Distance IQR:", distance_iqr)
print("Lower limit:", distance_lower)
print("Upper limit:", distance_upper)

# Count outliers
distance_outliers = (
    (df["distance_km"] < distance_lower) |
    (df["distance_km"] > distance_upper)
)

print("\nDistance outliers:", distance_outliers.sum())


Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Lower limit: -7.9375
Upper limit: 25.5625

Distance outliers: 26


In [ ]:

# Cap distance values within the IQR limits
df["distance_km"] = np.clip(
    df["distance_km"],
    distance_lower,
    distance_upper
)

# Round distance to 1 decimal place
df["distance_km"] = df["distance_km"].round(1)

print("Distance outliers capped successfully!")
print("Maximum distance:", df["distance_km"].max())


Distance outliers capped successfully!
Maximum distance: 25.6


In [ ]:

# Calculate quartiles for consultation fees
fee_q1 = df["fee_inr"].quantile(0.25)
fee_q3 = df["fee_inr"].quantile(0.75)

# Calculate IQR
fee_iqr = fee_q3 - fee_q1

# Calculate lower and upper limits
fee_lower = fee_q1 - (1.5 * fee_iqr)
fee_upper = fee_q3 + (1.5 * fee_iqr)

print("Fee Q1:", fee_q1)
print("Fee Q3:", fee_q3)
print("Fee IQR:", fee_iqr)
print("Lower limit:", fee_lower)
print("Upper limit:", fee_upper)

# Count outliers
fee_outliers = (
    (df["fee_inr"] < fee_lower) |
    (df["fee_inr"] > fee_upper)
)

print("\nFee outliers:", fee_outliers.sum())


Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Lower limit: 68.75
Upper limit: 1218.75

Fee outliers: 3


In [ ]:

# Cap consultation fees within the IQR limits
df["fee_inr"] = np.clip(
    df["fee_inr"],
    fee_lower,
    fee_upper
)

# Round fees to whole rupees
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Fee outliers capped successfully!")
print("Maximum fee:", df["fee_inr"].max())


Fee outliers capped successfully!
Maximum fee: 1219


In [ ]:

# Calculate the number of days between booking and appointment
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days                                                           #.dt.days extracts the number of days from the difference between the two dates

# Display the first 5 records
display(df[["appointment_id", "booked_on", "appt_date", "lead_days"]].head())

print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())


,appointment_id,booked_on,appt_date,lead_days
0,A0001,2025-01-24,2025-01-25,1
1,A0002,2025-05-27,2025-06-06,10
2,A0003,2025-03-24,2025-03-30,6
4,A0004,2025-04-18,2025-04-22,4
5,A0005,2024-11-23,2025-01-02,40


Minimum lead days: 1
Maximum lead days: 45


In [ ]:

# Correctly classify No-Show appointments
df["no_show"] = (df["status"] == "No-Show").astype(int)           # It converts boolean types to numbers : True becomes 1 ............. False becomes 0

# Check the counts
print("No-show counts:")
print(df["no_show"].value_counts())

# Calculate no-show percentage
no_show_rate = df["no_show"].mean() * 100

print("\nNo-show rate:", round(no_show_rate, 1), "%")


No-show counts:
no_show
0    772
1    218
Name: count, dtype: int64

No-show rate: 22.0 %


## 1.4 — Feature Engineering and NumPy Summary

In [ ]:

# Select numerical columns for analysis
columns = ["age", "distance_km", "fee_inr", "lead_days"]

print("NUMPY SUMMARY")
print("-------------------------")

columns = ["age", "distance_km", "fee_inr", "lead_days"]

X = df[columns].to_numpy()

print("Column means:", np.mean(X, axis=0))
print("Column medians:", np.median(X, axis=0))
print("Column standard deviations:", np.std(X, axis=0))


NUMPY SUMMARY
-------------------------
Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Column medians: [ 48.    8.2 650.   10. ]
Column standard deviations: [ 16.6469616    6.15720233 183.70963613   8.73369392]


In [ ]:

# Identify appointments with lead time greater than 14 days
long_lead = df["lead_days"] > 14

print("Appointments booked more than 14 days ahead:",
      long_lead.sum())

print("Percentage:",
      round(long_lead.mean() * 100, 1), "%")


Appointments booked more than 14 days ahead: 296
Percentage: 29.9 %


In [ ]:
'''
A z-score tells us how far a value is from the average, measured in standard deviations.

Formula:

z = {x-mu}/{sigma}


Where:
- x= individual distance
- mu= average distance
- sigma= standard deviation
'''


# Convert distance column into a NumPy array
distances = df["distance_km"].to_numpy()

# Calculate mean and standard deviation
distance_mean = np.mean(distances)
distance_std = np.std(distances)

# Calculate z-scores
distance_z = (distances - distance_mean) / distance_std

# Count unusually distant appointments
unusual_distances = np.abs(distance_z) > 2

print("Average distance:", round(distance_mean, 2))
print("Distance standard deviation:", round(distance_std, 2))

print("Appointments with absolute z-score above 2:",
      unusual_distances.sum())


Average distance: 9.5
Distance standard deviation: 6.16
Appointments with absolute z-score above 2: 53


In [ ]:

import pandas as pd

# Load clinics data
clinics = pd.read_csv("/content/clinics.csv")

# Display clinic records
print("Clinics dataset:")
display(clinics)

# Check columns and data types
print("\nDataset information:")
clinics.info()


Clinics dataset:


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   clinic_id    8 non-null      object 
 1   clinic_name  8 non-null      object 
 2   city         8 non-null      object 
 3   tier         8 non-null      object 
 4   latitude     8 non-null      float64
 5   longitude    8 non-null      float64
dtypes: float64(2), object(4)
memory usage: 516.0+ bytes


In [ ]:
print("Available columns:")
print(clinics.columns.tolist())

Available columns:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


## 1.5 — Enrich Clinic Data Using an API



In [ ]:

import requests

def get_temperature(latitude, longitude, clinic_name):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    reason = "Unknown error"

    # First attempt + one retry
    for attempt in range(2):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status code: {response.status_code}"
                continue

            data = response.json()

            hourly = data.get("hourly", {})
            temperatures = hourly.get("temperature_2m", [])

            if temperatures and temperatures[0] is not None:
                return temperatures[0]

            reason = "Hourly temperature data is missing"

        except requests.exceptions.RequestException as error:
            reason = f"Request failed: {error}"

        except (ValueError, TypeError, KeyError) as error:
            reason = f"Invalid API response: {error}"

    print(f"{clinic_name}: {reason}")
    return None


# Fetch temperatures for all eight clinics
clinics["temp_c"] = clinics.apply(
    lambda row: get_temperature(
        row["latitude"],
        row["longitude"],
        row["clinic_name"]
    ),
    axis=1
)

print("Clinic weather enrichment completed.")
display(clinics)



Clinic weather enrichment completed.


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.2
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,18.6
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,14.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,14.4
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,27.9
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,16.6
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,14.0


In [ ]:
clinics.to_csv(
    "/content/clinics_enriched.csv",
    index=False
)

print("Saved successfully: clinics_enriched.csv")

Saved successfully: clinics_enriched.csv


In [ ]:
print("Dataset shape:", clinics.shape)
print("Missing temperatures:", clinics["temp_c"].isna().sum())

print("\nTemperature for each clinic:")
print(clinics[["clinic_name", "temp_c"]])

Dataset shape: (8, 7)
Missing temperatures: 0

Temperature for each clinic:
                clinic_name  temp_c
0     Kothrud Family Clinic    19.2
1      Hadapsar Care Centre    19.5
2    Sitabuldi Health Point    18.6
3      Malviya Nagar Clinic    14.4
4     Vaishali Nagar Clinic    14.4
5  Edappally Medical Centre    27.9
6        Vijay Nagar Clinic    16.6
7    Gomti Nagar Care Point    14.0


### 1.6 Clean Table and SQL File Generation

In [ ]:

# Required column order from the assignment
required_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Make a separate copy of the cleaned dataset
clean_export = df[required_columns].copy()

# Sort by appointment_id
clean_export = clean_export.sort_values(
    by="appointment_id"
).reset_index(drop=True)

# Format both date columns as YYYY-MM-DD
clean_export["booked_on"] = (
    pd.to_datetime(clean_export["booked_on"])
    .dt.strftime("%Y-%m-%d")
)

clean_export["appt_date"] = (
    pd.to_datetime(clean_export["appt_date"])
    .dt.strftime("%Y-%m-%d")
)

# Validate the dataset BEFORE exporting
print("Dataset shape:", clean_export.shape)

print(
    "Unique appointment IDs:",
    clean_export["appointment_id"].nunique()
)

print(
    "Duplicate appointment IDs:",
    clean_export["appointment_id"].duplicated().sum()
)

print("\nMissing values by column:")
print(clean_export.isnull().sum())

# Stop if duplicate IDs or missing values are found
if clean_export["appointment_id"].duplicated().any():
    raise ValueError("Duplicate appointment IDs found!")

if clean_export.isnull().any().any():
    raise ValueError("Missing values found in the dataset!")

# Export only after validation passes
clean_export.to_csv(
    "/content/appointments_clean.csv",
    index=False
)

print("\nappointments_clean.csv saved successfully!")


Dataset shape: (990, 16)
Unique appointment IDs: 990
Duplicate appointment IDs: 0

Missing values by column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

appointments_clean.csv saved successfully!


In [ ]:
# Read the exported file again
check_df = pd.read_csv("/content/appointments_clean.csv")

print("Dataset shape:", check_df.shape)

print(
    "Unique appointment IDs:",
    check_df["appointment_id"].nunique()
)

print(
    "Missing values:",
    check_df.isnull().sum().sum()
)

print("\nColumn order:")
print(check_df.columns.tolist())

print("\nFirst five rows:")
display(check_df.head())

Dataset shape: (990, 16)
Unique appointment IDs: 990
Missing values: 0

Column order:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

First five rows:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:

import pandas as pd
import numpy as np

# Read the two exported CSV files
clinics_sql_df = pd.read_csv("/content/clinics_enriched.csv")
appointments_sql_df = pd.read_csv("/content/appointments_clean.csv")

# Convert Python values into SQL-compatible values
def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)

    text_value = str(value).replace("'", "''")
    return f"'{text_value}'"


sql_statements = []

# Insert clinics FIRST
for _, row in clinics_sql_df.iterrows():
    values = [sql_value(value) for value in row]
    statement = f"INSERT INTO clinics VALUES ({', '.join(values)});"
    sql_statements.append(statement)

# Insert appointments SECOND
for _, row in appointments_sql_df.iterrows():
    values = [sql_value(value) for value in row]
    statement = f"INSERT INTO appointments VALUES ({', '.join(values)});"
    sql_statements.append(statement)

# Save SQL statements
with open("/content/load_data.sql", "w", encoding="utf-8") as file:
    for statement in sql_statements:
        file.write(f"{statement}\n")

print("load_data.sql created successfully!")
print("Clinic INSERT statements:", len(clinics_sql_df))
print("Appointment INSERT statements:", len(appointments_sql_df))
print("Total INSERT statements:", len(sql_statements))


load_data.sql created successfully!
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998


In [ ]:
# Read the SQL file
with open("/content/load_data.sql", "r", encoding="utf-8") as file:
    sql_lines = file.readlines()

print("Total SQL statements:", len(sql_lines))

print("\nFirst 3 SQL statements:")
for line in sql_lines[:3]:
    print(line.strip())

print("\nLast 3 SQL statements:")
for line in sql_lines[-3:]:
    print(line.strip())

# Check statement order
clinic_statements = [
    line for line in sql_lines
    if line.startswith("INSERT INTO clinics")
]

appointment_statements = [
    line for line in sql_lines
    if line.startswith("INSERT INTO appointments")
]

print("\nClinic statements:", len(clinic_statements))
print("Appointment statements:", len(appointment_statements))

Total SQL statements: 998

First 3 SQL statements:
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 19.2);
INSERT INTO clinics VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.926, 19.5);
INSERT INTO clinics VALUES ('C03', 'Sitabuldi Health Point', 'Nagpur', 'Tier-2', 21.1458, 79.0882, 18.6);

Last 3 SQL statements:
INSERT INTO appointments VALUES ('A0998', 'P422', 'C06', '2025-05-05', '2025-05-16', 58, 'M', 3.2, 850, 'Walk-In', 'Consultation', 'No', 1, 'Attended', 11, 0);
INSERT INTO appointments VALUES ('A0999', 'P509', 'C05', '2025-03-30', '2025-04-12', 59, 'F', 25.6, 300, 'Walk-In', 'Follow-up', 'Yes', 0, 'Attended', 13, 0);
INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);

Clinic statements: 8
Appointment statements: 990
